# 分开检查问题、资料与回答依据

一条回答不理想，可能是问题本身不清楚，也可能是检索没有找到资料，或者回答没有正确使用已经找到的资料。把这些情况混成一个总分，很难知道该改哪里。

本节读取教程附带的 987 个已切分片段，并直接使用保存的 BGE 向量库。运行时只计算新问题的向量，不会重新切分 PDF，也不会重算文档向量，更不会把预期页或参考答案交给检索器。输出已经保存在 Notebook 中，直接打开即可阅读。

## 先确定检查顺序

1. **问题**：先由人工确认读者想问什么，问题是否缺少必要条件？
2. **资料**：需要的页有没有找到，排在第几位，必要内容是否完整？
3. **回答**：每个主要结论能否在资料中找到依据，标注的页码是否对应？

前一项有问题时，先处理前一项。例如目标页根本没有进入上下文，就不该先改回答用的 Prompt。

## 读取检索结果

运行时直接复用教程附带的 987 个片段和 BGE 向量库。如果环境没有准备好，代码会直接报错，不会换成另一种检索方法，以免把环境差异误当作实验结论。环境准备见[配套向量库说明](../data/向量库/README.md)。

In [1]:
import re
import sys
from pathlib import Path


def find_course_root(start: Path) -> Path:
    for folder in (start, *start.parents):
        if (folder / "data" / "dataset/manifest.json").is_file():
            return folder
    raise FileNotFoundError("没有找到教程数据目录，请从本节所在目录运行。")


course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import (
    load_query_catalog, load_default_collection, load_default_chunks, build_default_chunk_search,
)
from common.nontraining_utils import load_annotation

cases = {case["id"]: case for case in load_query_catalog()}
collection = load_default_collection()
chunks = load_default_chunks(collection)
search = build_default_chunk_search(collection)
search_kind = "教程随附的 BGE 向量库"
print(f"已读取配套片段，共 {len(chunks)} 个；检查方式：{search_kind}。")

已读取配套片段，共 987 个；检查方式：教程随附的 BGE 向量库。


In [2]:
selected_ids = [
    "business_model_selection",
    "svm_kernel_evidence",
    "lda_generalized_eigenvalue",
]

retrieval_records = []
for case_id in selected_ids:
    case = cases[case_id]
    results = search(case["query"], top_k=4)
    annotation = load_annotation(case_id)
    expected = set(annotation["expected_pages"])
    pages = [item.pages[0] for item in results]
    first_rank = next(
        (rank for rank, item in enumerate(results, 1) if expected.intersection(item.pages)),
        None,
    )
    found = sorted(expected.intersection(pages))
    text = " ".join(item.text for item in results)
    content_points = {
        "对偶形式中的样本内积": "内积" in text,
        "映射到更高维特征空间": "高维" in text,
    } if case_id == "svm_kernel_evidence" else {}
    retrieval_records.append({
        "case": case,
        "results": results,
        "found": found,
        "first_rank": first_rank,
        "question_status": "人工待确认：问题是否清楚、条件是否完整",
        "material_status": ("必要资料已找全" if len(found) == len(expected) else "必要资料未找全"),
        "content_points": content_points,
    })
    print(f"\n问题：{case['query']}")
    print(f"需要查阅的页：{sorted(expected)}")
    print(f"返回页：{pages}")
    print(f"找到：{len(found)}/{len(expected)}；首个目标排名：{first_rank or '前 4 个没有找到'}")


问题：如何判断模型好坏并挑选适合业务场景的模型？
需要查阅的页：[18]
返回页：[15, 16, 15, 18]
找到：1/1；首个目标排名：4



问题：SVM 为什么能用核函数处理原始空间线性不可分的问题？请同时说明对偶形式中的内积和高维映射。
需要查阅的页：[66]
返回页：[66, 66, 68, 74]
找到：1/1；首个目标排名：1

问题：在线性判别分析（LDA）中，为什么需要最大化类间散度与类内散度之比？这个优化问题是如何转化为广义特征值问题的？请结合南瓜书中的 LDA 推导步骤说明。
需要查阅的页：[41, 42, 43, 44]
返回页：[186, 139, 130, 68]
找到：0/4；首个目标排名：前 4 个没有找到


## 不同结果要改不同地方

- **业务模型选择**：第 18 页出现在前 4 个片段中，资料层暂时通过，但排名不算靠前，仍要检查片段内容是否真的回答问题。
- **SVM 核函数**：第 66 页在前 4 个片段中；下面再分别检查对偶内积和高维映射这两个回答要点。
- **LDA 推导**：需要连续几页，前 4 个片段没有找到必要页。此时先记录资料缺口，不要把它写成回答错误。

目标页只是一种便于复查的记录方式。即使找到了对应页码，也要打开片段确认内容是否足够。

In [3]:
for record in retrieval_records:
    case = record["case"]
    print(f"\n问题：{case['query']}")
    for rank, item in enumerate(record["results"], 1):
        readable = " ".join(re.sub(r"[\x00-\x1f\x7f]", " ", item.text).split())
        print(f"{rank}. 第 {item.pages[0]} 页：{readable[:160]}")
    if record["content_points"]:
        print("回答要点的资料层检查：", record["content_points"])


问题：如何判断模型好坏并挑选适合业务场景的模型？
1. 第 15 页：，因此模型学习到的经验也多，自然表现效果越好。例如以上举例中如果训练集中含有相同颜色但根蒂不蜷缩的坏瓜，模型a学到真相的概率则也会增大；从特征工程的角度来说，通常对特征数值化越合理，特征收集越全越细致，模型效果通常越好，因为此时模型更易学得样本之间潜在的规律。例如学习区分亚洲人和非洲人时，此时样本即
2. 第 16 页：习算法有不同的偏好，我们称为“归纳偏好”。对于当前房价预测这个例子来说，这两个算法学得的模型哪个更好呢？著名的“奥卡姆剃刀”原则认为“若有多个假设与观察一致，则选最简单的那个”，但是何为“简单”便见仁见智了，如果认为函数的幂次越低越简单，则此时一元线性回归算法更好，如果认为幂次越高越简单，则此时多项式回归算法更好，因此
3. 第 15 页：型对训练集中每个样本的判断都对，但是其所学到的规律是不同的。导致此现象最直接的原因是算法的不同，但是算法通常是有限的，可穷举的，尤其是在特定任务场景下可使用的算法更是有限，因此，数据便是导致此现象的另一重要原因，这也就是机器学习领域常说的“数据决定模型的上限，而算法则是让模型无限逼近上限”,下面详细解释此话的含义。先解
4. 第 18 页：第2章模型评估与选择如“西瓜书”前言所述，本章仍属于机器学习基础知识，如果说第1章介绍了什么是机器学习及机器学习的相关数学符号，那么本章则进一步介绍机器学习的相关概念。具体来说，介绍内容正如本章名称“模型评估与选择”所述，讲述的是如何评估模型的优劣和选择最适合自己业务场景的模型。由于“模型评估与选择”是在模型产出以后进

问题：SVM 为什么能用核函数处理原始空间线性不可分的问题？请同时说明对偶形式中的内积和高维映射。
1. 第 66 页：(6.22)的解释此即核函数的定义，即核函数可以分解成两个向量的内积。要想了解某个核函数是如何将原始特征空间映射到更高维的特征空间的，只需要分解为两个表达形式完全一样的向量内积即可。6.4软间隔与正则化6.4.1式(6.35)令max 0,1−yi wTxi+b =ξi显然ξi≥0，且当1−yi wTxi+b >0时有
2. 第 66 页：(1)式(6.6)中的未知数是w和b，式(6.11)中的未知数是α，w的维度d对应样本特征个数，α的维度m对应训练样本个数，通常m≪d，

## 回答依据需要单独检查

检索结果合格后，再逐条读回答：

- 把回答拆成几个可以核对的主要结论；
- 为每个结论指出支持它的片段和页码；
- 找不到原文依据的内容，删掉、改成不确定表述，或继续查资料；
- 再看回答是否真正回应了问题。

下面只建立记录表，不生成回答，也不用问题集中的参考答案代替模型输出。问题栏保留“人工待确认”记录，这不是自动检查结果；回答栏明确标为“未生成”，所以这里没有进行回答正确性或引用正确性验证。读者需要在真正生成回答后，再把每个结论回到片段逐条核对。

In [4]:
review_rows = []
for record in retrieval_records:
    case = record["case"]
    review_rows.append({
        "问题": case["query"],
        "问题检查": record["question_status"],
        "资料检查": record["material_status"],
        "需要的页": load_annotation(case["id"])["expected_pages"],
        "已找到的页": record["found"],
        "回答检查": "未生成回答；尚未验证原文支持或引用",
    })

for row in review_rows:
    print(row)

{'问题': '如何判断模型好坏并挑选适合业务场景的模型？', '问题检查': '人工待确认：问题是否清楚、条件是否完整', '资料检查': '必要资料已找全', '需要的页': [18], '已找到的页': [18], '回答检查': '未生成回答；尚未验证原文支持或引用'}
{'问题': 'SVM 为什么能用核函数处理原始空间线性不可分的问题？请同时说明对偶形式中的内积和高维映射。', '问题检查': '人工待确认：问题是否清楚、条件是否完整', '资料检查': '必要资料已找全', '需要的页': [66], '已找到的页': [66], '回答检查': '未生成回答；尚未验证原文支持或引用'}
{'问题': '在线性判别分析（LDA）中，为什么需要最大化类间散度与类内散度之比？这个优化问题是如何转化为广义特征值问题的？请结合南瓜书中的 LDA 推导步骤说明。', '问题检查': '人工待确认：问题是否清楚、条件是否完整', '资料检查': '必要资料未找全', '需要的页': [41, 42, 43, 44], '已找到的页': [], '回答检查': '未生成回答；尚未验证原文支持或引用'}


## 小结

先确定问题是否清楚，再看资料有没有找全，再检查回答是否使用了资料。这里没有计算总分，因为“检索没找全”和“回答没有依据”需要完全不同的处理。

本次输出中的问题检查全部写成“人工待确认”，所以不能把它理解成问题已经自动验证；回答检查全部写成“未生成”，也不能把它理解成回答已验证。最终仍要由人回到问题和原文逐条核对。



## 为什么三份检查不能合成一个总分

问题检查确认用户意图、范围和条件；资料检查确认必要内容是否进入上下文、排名是否足够靠前、片段是否被截断；回答检查逐条拆分可核对的结论，核对原文支持和引用页。资料没找全时先修检索，不能用一个较强的回答提示词掩盖召回失败；资料已找全但结论无依据时才改生成规则。

本页用“如何判断模型好坏并挑选适合业务场景的模型”“SVM 为什么能用核函数处理原始空间线性不可分的问题？”和“在线性判别分析中，为什么要把散度比转成广义特征值问题？”三个问题做对照；这里只展示检查流程，不把它们包装成某个增强方法的两道实验题。


## 三份记录各自回答什么问题

问题层只检查用户意图是否明确：主体、范围、时间和必要条件有没有缺失；资料层检查需要的证据是否在返回结果中、排名是否足够靠前、片段是否被截断；回答层把回答拆成可核对的结论，逐条记录支持它的原文和页码。三层要分开记录，不能用资料命中率代替回答正确率，也不能把回答流畅当成问题已经清楚。

空白项表示“尚未人工判断”，不是通过。它只生成一张待复核记录，不会把期望页或参考答案传给检索器。

In [5]:
def make_three_part_review(case_id, query, returned_pages, expected_pages=(), claims=(), *,
                          question_clear=None, scope_complete=None):
    """生成待人工复核的 Query/Context/Response 记录。"""
    returned_pages = list(returned_pages)
    expected_pages = set(expected_pages)
    found_pages = sorted(expected_pages.intersection(returned_pages))
    claim_rows = list(claims)
    return {
        "case_id": case_id,
        "query": {
            "text": query,
            "clear": question_clear,
            "scope_complete": scope_complete,
        },
        "context": {
            "returned_pages": returned_pages,
            "expected_pages_for_review": sorted(expected_pages),
            "found_pages": found_pages,
            "first_expected_rank": next((i for i, page in enumerate(returned_pages, 1)
                                    if page in expected_pages), None),
        },
        "response": {
            "claims": claim_rows,
            "supported_claims": sum(bool(row.get("supported")) for row in claim_rows),
            "claim_count": len(claim_rows),
        },
    }

# 例：问题层和回答层仍留给人工填写；这里不调用模型，也不改变前面的保存输出。
review_template = make_three_part_review(
    "lda_generalized_eigenvalue",
    "在线性判别分析中，散度比怎样转成广义特征值问题？",
    returned_pages=[44, 131, 126, 139],
    expected_pages=[41, 42, 43, 44],
    claims=[{"text": "SbW=λSwW", "supported": None, "evidence": []}],
)
assert review_template["query"]["clear"] is None
assert review_template["response"]["supported_claims"] == 0
print(
    "示例复核：已找到资料页 "
    + "、".join(str(page) for page in review_template["context"]["found_pages"])
    + "；问题状态：待人工确认；回答状态：未生成。"
)

示例复核：已找到资料页 44；问题状态：待人工确认；回答状态：未生成。
